In [1]:
def load_data():
    import pandas as pd
    import numpy as np
    data_url = "http://lib.stat.cmu.edu/datasets/boston"
    raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
    # now we split the data into data and target
    data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
    target = raw_df.values[1::2, 2]
    # These are feature names
    feature_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']
    # create a data frame
    df = pd.DataFrame(data, columns=feature_names)
    df['MEDV'] = target # here MEDV is our target variable
    
    return df

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Prabhakar\AppData\Local\Temp\ipykernel_70596\1283889103.py:5: SyntaxWarning: invalid escape sequence '\s'
  raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)


In [7]:
df = load_data()
print(df.head())

      CRIM    ZN  INDUS  CHAS    NOX     RM   AGE     DIS  RAD    TAX  \
0  0.00632  18.0   2.31   0.0  0.538  6.575  65.2  4.0900  1.0  296.0   
1  0.02731   0.0   7.07   0.0  0.469  6.421  78.9  4.9671  2.0  242.0   
2  0.02729   0.0   7.07   0.0  0.469  7.185  61.1  4.9671  2.0  242.0   
3  0.03237   0.0   2.18   0.0  0.458  6.998  45.8  6.0622  3.0  222.0   
4  0.06905   0.0   2.18   0.0  0.458  7.147  54.2  6.0622  3.0  222.0   

   PTRATIO       B  LSTAT  MEDV  
0     15.3  396.90   4.98  24.0  
1     17.8  396.90   9.14  21.6  
2     17.8  392.83   4.03  34.7  
3     18.7  394.63   2.94  33.4  
4     18.7  396.90   5.33  36.2  


In [8]:
# Train & Evaluate
# ----------------------------
def evaluate_models():
    from pprint import pprint
    from sklearn.model_selection import train_test_split, GridSearchCV
    from sklearn.metrics import mean_squared_error, r2_score
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LinearRegression, Ridge
    from sklearn.ensemble import RandomForestRegressor

    # Load
    df = load_data()
    X = df.drop(columns=['MEDV']).values
    y = df['MEDV'].values

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Pipelines
    pipe_lr = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("model", LinearRegression())
    ])
    pipe_ridge = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("model", Ridge())
    ])
    pipe_rf = Pipeline([
        ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
    ])

    # EXACTLY 3 HYPERPARAMETERS EACH
    param_grid_lr = {
        "model__fit_intercept": [True, False],
        "model__copy_X": [True, False],
        "model__positive": [False, True],
    }

    param_grid_ridge = {
        "model__alpha": [0.1, 1.0, 10.0, 100.0],
        "model__solver": ["auto", "svd", "cholesky", "lsqr"],
        "model__fit_intercept": [True, False],
    }

    param_grid_rf = {
        "model__n_estimators": [200, 400, 800],
        "model__max_depth": [None, 5, 10, 20],
        "model__max_features": ["sqrt", "log2", None],
    }

    # Grid searches
    searches = {
        "LinearRegression": GridSearchCV(
            estimator=pipe_lr,
            param_grid=param_grid_lr,
            scoring="neg_mean_squared_error",
            cv=5,
            n_jobs=-1
        ),
        "Ridge": GridSearchCV(
            estimator=pipe_ridge,
            param_grid=param_grid_ridge,
            scoring="neg_mean_squared_error",
            cv=5,
            n_jobs=-1
        ),
        "RandomForest": GridSearchCV(
            estimator=pipe_rf,
            param_grid=param_grid_rf,
            scoring="neg_mean_squared_error",
            cv=5,
            n_jobs=-1
        ),
    }

    # Fit & evaluate
    results = []
    best_params_out = {}
    for name, search in searches.items():
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        y_pred = best_model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        results.append((name, mse, r2))
        best_params_out[name] = search.best_params_

    # Print results
    print("Model Performance on Boston Housing Dataset (with Hyperparameter Tuning)")
    print("{:<15} {:<15} {:<15}".format("Model", "MSE", "R²"))
    for name, mse, r2 in results:
        print("{:<15} {:<15.3f} {:<15.3f}".format(name, mse, r2))

    print("\nBest hyperparameters found (via 5-fold CV on train set):")
    pprint(best_params_out)


if __name__ == "__main__":
    # quick peek
    df = load_data()
    print(df.head())
    # run tuning + evaluation
    evaluate_models()

      CRIM    ZN  INDUS  CHAS    NOX     RM   AGE     DIS  RAD    TAX  \
0  0.00632  18.0   2.31   0.0  0.538  6.575  65.2  4.0900  1.0  296.0   
1  0.02731   0.0   7.07   0.0  0.469  6.421  78.9  4.9671  2.0  242.0   
2  0.02729   0.0   7.07   0.0  0.469  7.185  61.1  4.9671  2.0  242.0   
3  0.03237   0.0   2.18   0.0  0.458  6.998  45.8  6.0622  3.0  222.0   
4  0.06905   0.0   2.18   0.0  0.458  7.147  54.2  6.0622  3.0  222.0   

   PTRATIO       B  LSTAT  MEDV  
0     15.3  396.90   4.98  24.0  
1     17.8  396.90   9.14  21.6  
2     17.8  392.83   4.03  34.7  
3     18.7  394.63   2.94  33.4  
4     18.7  396.90   5.33  36.2  
Model Performance on Boston Housing Dataset (with Hyperparameter Tuning)
Model           MSE             R²             
LinearRegression 24.291          0.669          
Ridge           24.313          0.668          
RandomForest    10.177          0.861          

Best hyperparameters found (via 5-fold CV on train set):
{'LinearRegression': {'model__cop